# Module 7: Regime-Aware Feature Selection

## Learning Objectives

By completing this notebook, you will:
1. Fit a Gaussian HMM to detect market regimes from financial returns
2. Run feature selection separately within each detected regime
3. Quantify how much the optimal feature sets differ across regimes (Jaccard overlap)
4. Implement adaptive re-selection triggered by regime change
5. Compare regime-aware vs static feature selection performance

## Prerequisites
- Module 07 Guide 03: Regime-aware selection concepts
- Notebook 01: Granger feature ranking

## Estimated Time: 15 minutes

## Dataset
We use US equity market data downloaded from Yahoo Finance via `yfinance`. We use:
- **Target:** S&P 500 monthly returns (excess over T-bill)
- **Features:** 20 macro/market variables including VIX, yield spread, momentum, credit spread

If the internet is unavailable, we fall back to a realistic synthetic dataset.

## 1. Setup and Data Loading

In [ ]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# HMM library
from hmmlearn.hmm import GaussianHMM

# Feature selection and modelling
from sklearn.linear_model import Lasso, Ridge
from sklearn.feature_selection import mutual_info_regression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from statsmodels.tsa.stattools import grangercausalitytests

# Drift detection
from scipy.stats import ks_2samp, wasserstein_distance

# Visualisation
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
np.random.seed(42)

print('Libraries loaded.')

### 1.1 Load Market Data

In [ ]:
# Purpose: Load equity market data with macro features
# Using yfinance for real data; synthetic fallback if unavailable

try:
    import yfinance as yf

    # Download S&P 500 and CBOE VIX index
    sp500 = yf.download('^GSPC', start='1990-01-01', end='2024-12-31',
                        interval='1mo', progress=False)['Close']
    vix = yf.download('^VIX', start='1990-01-01', end='2024-12-31',
                      interval='1mo', progress=False)['Close']

    # Compute monthly log-returns
    sp500_ret = np.log(sp500).diff().dropna()
    sp500_ret.name = 'SPX_return'

    # Align
    common = sp500_ret.index.intersection(vix.index)
    target = sp500_ret.loc[common]
    vix_aligned = vix.loc[common]

    print(f'Loaded S&P 500 and VIX: {len(target)} monthly observations')
    print(f'Date range: {target.index[0].date()} to {target.index[-1].date()}')
    data_source = 'yfinance'

except Exception as e:
    print(f'yfinance unavailable ({e}). Generating synthetic market data.')
    data_source = 'synthetic'

if data_source == 'synthetic':
    # Synthetic market data with two clear regimes
    T = 400
    dates = pd.date_range('1990-01-31', periods=T, freq='ME')
    rng = np.random.default_rng(42)

    # Regime 0: low-vol bull (mean=0.008, vol=0.03)
    # Regime 1: high-vol bear (mean=-0.012, vol=0.08)
    regime_true = np.zeros(T, dtype=int)
    # Create realistic regime sequences
    t = 0
    while t < T:
        if regime_true[max(0, t-1)] == 0:
            # Stay in regime 0 for ~30 months on average
            dur = int(rng.exponential(30))
        else:
            dur = int(rng.exponential(10))
        end = min(t + dur, T)
        regime_true[t:end] = regime_true[max(0, t-1)]
        if end < T:
            regime_true[end] = 1 - regime_true[max(0, t-1)]
        t = end + 1

    returns = np.where(
        regime_true == 0,
        rng.normal(0.008, 0.03, T),
        rng.normal(-0.012, 0.08, T)
    )
    target = pd.Series(returns, index=dates, name='SPX_return')

    # Simulate VIX-like series
    vix_values = np.where(regime_true == 0,
                          rng.normal(15, 3, T),
                          rng.normal(30, 8, T)).clip(5, 80)
    vix_aligned = pd.Series(vix_values, index=dates, name='VIX')

    print(f'Synthetic market data: {T} months')
    print(f'Regime 0 (low-vol): {(regime_true==0).sum()} months ({(regime_true==0).mean():.0%})')
    print(f'Regime 1 (high-vol): {(regime_true==1).sum()} months ({(regime_true==1).mean():.0%})')

### 1.2 Construct Feature Set

We build 20 candidate features representing different aspects of market conditions:
momentum, volatility, yield curve, and lagged returns.

In [ ]:
# Purpose: Construct realistic macro/market features
# Each feature captures a different aspect of market state

T = len(target)
dates = target.index
rng_feat = np.random.default_rng(123)

feature_dict = {}

# VIX (if loaded from yfinance, use it; otherwise from synthetic above)
feature_dict['VIX'] = vix_aligned.reindex(dates).fillna(method='ffill')
feature_dict['VIX_change'] = feature_dict['VIX'].diff()
feature_dict['VIX_zscore'] = (
    (feature_dict['VIX'] - feature_dict['VIX'].rolling(12).mean()) /
    feature_dict['VIX'].rolling(12).std()
)

# Momentum features (lagged returns)
for lag in [1, 2, 3, 6, 12]:
    feature_dict[f'ret_lag{lag}m'] = target.shift(lag)

# Rolling volatility
for window in [3, 6, 12]:
    feature_dict[f'realvol_{window}m'] = target.rolling(window).std()

# Trend indicator: ratio of 3m to 12m momentum
feature_dict['trend_ratio'] = (
    target.shift(1).rolling(3).mean() /
    target.shift(1).rolling(12).mean().abs().clip(lower=0.001)
)

# Yield-curve proxy (simulated in absence of FRED data)
# In real usage: 10yr - 3m Treasury spread from FRED
yield_spread = pd.Series(
    rng_feat.normal(1.5, 1.2, T) + 0.3 * target.fillna(0).cumsum().values * 0.01,
    index=dates, name='yield_spread'
).clip(-2, 5)
feature_dict['yield_spread'] = yield_spread
feature_dict['yield_spread_lag1'] = yield_spread.shift(1)

# Credit spread proxy (simulated)
credit_spread = pd.Series(
    rng_feat.normal(2.0, 0.8, T) + 0.5 * feature_dict['VIX'].values / 20,
    index=dates, name='credit_spread'
).clip(0.5, 8)
feature_dict['credit_spread'] = credit_spread
feature_dict['credit_spread_chg'] = credit_spread.diff()

# Build DataFrame
features = pd.DataFrame(feature_dict).dropna()
target_aligned = target.loc[features.index]

print(f'Feature set: {features.shape[1]} features, {len(target_aligned)} observations')
print(f'Features: {list(features.columns)}')

---

## 2. HMM Regime Detection

Fit a 2-state Gaussian HMM to the return series. The HMM learns:
- State means and variances (characterising each regime)
- Transition probabilities between states
- Most likely state sequence (Viterbi decoding)

In [ ]:
# Purpose: Fit Gaussian HMM to detect market regimes
# Key: Use returns and volatility as inputs for richer regime characterisation

N_STATES = 2

# Input to HMM: returns (level) and rolling volatility (for better regime separation)
hmm_input = pd.concat([
    target_aligned.rename('returns'),
    target_aligned.rolling(3).std().rename('vol_3m'),
], axis=1).dropna()

X_hmm = hmm_input.values

# Fit HMM with multiple random restarts to find best solution
best_model = None
best_score = -np.inf

for seed in range(10):
    model = GaussianHMM(
        n_components=N_STATES,
        covariance_type='full',
        n_iter=300,
        random_state=seed,
        tol=1e-4,
    )
    model.fit(X_hmm)
    score = model.score(X_hmm)
    if score > best_score:
        best_score = score
        best_model = model

# Decode regimes
states_raw = best_model.predict(X_hmm)
posteriors = best_model.predict_proba(X_hmm)

# Label regimes by mean return (regime 0 = higher-return state)
state_means = [target_aligned.loc[hmm_input.index][states_raw == k].mean()
               for k in range(N_STATES)]
# Relabel: regime 0 = bull (higher mean), regime 1 = bear (lower mean)
sort_order = np.argsort(state_means)[::-1]  # highest mean first
label_map = {sort_order[i]: i for i in range(N_STATES)}
states = np.array([label_map[s] for s in states_raw])

regime_series = pd.Series(states, index=hmm_input.index, name='regime')
ret_aligned = target_aligned.loc[hmm_input.index]

# Regime statistics
print('HMM Regime Detection Results')
print('='*55)
for k in range(N_STATES):
    mask = regime_series == k
    r = ret_aligned[mask]
    print(f'\nRegime {k} ({"Bull" if k==0 else "Bear"}):')
    print(f'  Observations: {mask.sum()} ({mask.mean():.0%} of total)')
    print(f'  Mean monthly return: {r.mean():.4f} ({r.mean()*12:.2%} annualised)')
    print(f'  Monthly volatility:  {r.std():.4f} ({r.std()*np.sqrt(12):.2%} annualised)')
    print(f'  Sharpe (approx):     {r.mean()/r.std():.3f}')

print(f'\nHMM log-likelihood: {best_score:.2f}')
print(f'\nTransition probabilities:')
# Relabel transition matrix
P = best_model.transmat_
P_relabelled = np.zeros_like(P)
for i in range(N_STATES):
    for j in range(N_STATES):
        P_relabelled[label_map[i], label_map[j]] = P[i, j]
for i in range(N_STATES):
    print(f'  Regime {i} -> Regime 0: {P_relabelled[i,0]:.3f}, Regime 1: {P_relabelled[i,1]:.3f}')

In [ ]:
# Purpose: Visualise HMM regime labels overlaid on return series
# Key: Confirm that detected regimes correspond to economically meaningful periods

fig, axes = plt.subplots(3, 1, figsize=(15, 10), sharex=True)

regime_colors = {0: '#2ECC71', 1: '#E74C3C'}  # green=bull, red=bear
regime_names = {0: 'Bull (Regime 0)', 1: 'Bear (Regime 1)'}

# --- Panel 1: Returns with regime background shading ---
axes[0].bar(ret_aligned.index, ret_aligned.values,
            color=[regime_colors[r] for r in regime_series.values],
            width=20, alpha=0.8)
axes[0].axhline(0, color='black', linewidth=0.5)
axes[0].set_ylabel('Monthly Return')
axes[0].set_title('S&P 500 Monthly Returns — Coloured by HMM Regime', fontsize=12)
handles = [mpatches.Patch(color=regime_colors[k], label=regime_names[k])
           for k in range(N_STATES)]
axes[0].legend(handles=handles, loc='upper left')

# --- Panel 2: VIX overlaid with regime shading ---
vix_plot = features['VIX'].reindex(ret_aligned.index).dropna()
axes[1].plot(vix_plot.index, vix_plot.values, color='navy', linewidth=1)
for k in range(N_STATES):
    regime_periods = regime_series[regime_series == k].index
    axes[1].fill_between(ret_aligned.index,
                          0, vix_plot.max() * 1.1,
                          where=regime_series.reindex(ret_aligned.index).fillna(0) == k,
                          color=regime_colors[k], alpha=0.15)
axes[1].set_ylabel('VIX')
axes[1].set_title('VIX Index — Background Shading by Regime', fontsize=11)

# --- Panel 3: Posterior probability of being in Regime 1 (bear) ---
post_bear = pd.Series(
    posteriors[:, label_map.get(1, 1) if 1 in label_map.values() else 0],
    index=hmm_input.index
)
axes[2].fill_between(post_bear.index, 0, post_bear.values,
                     color='#E74C3C', alpha=0.5)
axes[2].axhline(0.5, color='black', linestyle='--', linewidth=1, label='Decision threshold')
axes[2].set_ylabel('P(Bear Regime)')
axes[2].set_xlabel('Date')
axes[2].set_title('HMM Posterior Probability: Bear Regime', fontsize=11)
axes[2].legend()
axes[2].set_ylim(0, 1)

plt.tight_layout()
plt.savefig('hmm_regime_detection.png', dpi=120, bbox_inches='tight')
plt.show()
print('Figure saved: hmm_regime_detection.png')

---

## 3. Regime-Specific Feature Selection

Select features separately within each detected regime.
We use both mutual information and LASSO to get two perspectives.

In [ ]:
# Purpose: Run feature selection independently within each regime
# Key: Different features matter in bull vs bear markets

TOP_K_REGIME = min(8, features.shape[1] - 1)

def select_features_in_regime(
    target: pd.Series,
    features: pd.DataFrame,
    regime_mask: pd.Series,
    top_k: int = 8,
    method: str = 'lasso',
) -> dict:
    """
    Feature selection restricted to observations in a specific regime.
    Uses lagged features to ensure temporal validity.
    """
    # Filter to regime observations
    regime_idx = regime_mask[regime_mask].index
    target_r = target.loc[regime_idx]

    # Use lag-1 features to predict current returns
    features_lag1 = features.shift(1).loc[regime_idx]

    common = target_r.dropna().index.intersection(features_lag1.dropna(how='all').index)
    y = target_r.loc[common]
    X = features_lag1.loc[common].fillna(0)

    if len(y) < 15:
        return {'selected': [], 'scores': pd.Series(0, index=features.columns)}

    scaler = StandardScaler()
    X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns, index=X.index)

    if method == 'mutual_info':
        scores = mutual_info_regression(X_scaled.values, y.values, random_state=42)
        score_series = pd.Series(scores, index=features.columns)

    elif method == 'lasso':
        lasso = Lasso(alpha=0.05, max_iter=3000, random_state=42)
        lasso.fit(X_scaled.values, y.values)
        score_series = pd.Series(np.abs(lasso.coef_), index=features.columns)

    top_features = score_series.nlargest(top_k).index.tolist()

    return {
        'selected': top_features,
        'scores': score_series,
        'n_obs': len(y),
    }


# Align features and regime labels
common_idx = features.index.intersection(regime_series.index).intersection(target_aligned.index)
features_aligned = features.loc[common_idx]
target_final = target_aligned.loc[common_idx]
regime_aligned = regime_series.loc[common_idx]

# Select features for each regime
print('Regime-Specific Feature Selection (LASSO, lag-1 features):')
print('='*60)

regime_selections = {}
regime_scores = {}

for k in range(N_STATES):
    mask = regime_aligned == k
    label = 'Bull' if k == 0 else 'Bear'

    result = select_features_in_regime(
        target_final, features_aligned, mask,
        top_k=TOP_K_REGIME, method='lasso'
    )
    regime_selections[k] = result['selected']
    regime_scores[k] = result['scores']

    print(f'\nRegime {k} ({label}) — {result["n_obs"]} observations:')
    print(f'  Top {TOP_K_REGIME} features: {result["selected"]}')

# Also: global feature selection (ignoring regime)
global_result = select_features_in_regime(
    target_final, features_aligned, pd.Series(True, index=common_idx),
    top_k=TOP_K_REGIME, method='lasso'
)
regime_selections['global'] = global_result['selected']
regime_scores['global'] = global_result['scores']

print(f'\nGlobal (regime-unaware) selection:')
print(f'  Top {TOP_K_REGIME} features: {global_result["selected"]}')

In [ ]:
# Purpose: Quantify regime feature set divergence using Jaccard similarity
# Key: Low Jaccard = regimes need different features = regime-awareness is valuable

def jaccard_similarity(set1: set, set2: set) -> float:
    """Jaccard similarity = |intersection| / |union|"""
    if len(set1 | set2) == 0:
        return 0.0
    return len(set1 & set2) / len(set1 | set2)

print('Feature Set Overlap Analysis')
print('='*50)

set0 = set(regime_selections[0])
set1 = set(regime_selections[1])
set_global = set(regime_selections['global'])

j_01 = jaccard_similarity(set0, set1)
j_0g = jaccard_similarity(set0, set_global)
j_1g = jaccard_similarity(set1, set_global)

print(f'Jaccard(Bull, Bear):   {j_01:.3f}  (0=no overlap, 1=identical)')
print(f'Jaccard(Bull, Global): {j_0g:.3f}')
print(f'Jaccard(Bear, Global): {j_1g:.3f}')

print(f'\nFeatures unique to Bull regime: {sorted(set0 - set1)}')
print(f'Features unique to Bear regime: {sorted(set1 - set0)}')
print(f'Features in both regimes: {sorted(set0 & set1)}')
print(f'Features in global but neither regime: {sorted(set_global - set0 - set1)}')

if j_01 < 0.4:
    print(f'\nConclusion: Jaccard({j_01:.2f}) < 0.4 — regimes require substantially different feature sets.')
    print('Regime-aware selection is justified.')
else:
    print(f'\nConclusion: Jaccard({j_01:.2f}) >= 0.4 — feature sets are moderately similar.')
    print('Some regime-specific adaptation is still valuable for edge features.')

---

## 4. Adaptive Re-Selection Triggered by Regime Change

Implement a simple production-style adaptive controller that:
1. Monitors the HMM posterior probability of regime change
2. Triggers re-selection when a regime transition is detected
3. Switches to the appropriate regime-specific feature set

In [ ]:
# Purpose: Adaptive regime-triggered feature re-selection
# Key: Only re-select when a regime transition is detected, reducing compute

REGIME_CHANGE_THRESHOLD = 0.70  # Trigger when P(new regime) > 0.70

def simulate_adaptive_selection(
    target: pd.Series,
    features: pd.DataFrame,
    regime_series: pd.Series,
    posteriors: np.ndarray,
    regime_selections: dict,
    min_train: int = 60,
    regime_threshold: float = 0.70,
) -> pd.DataFrame:
    """
    Simulate adaptive feature selection over time.

    At each time step t:
    - Check if regime has changed (posterior > threshold)
    - If yes: re-select features for new regime using data [0, t]
    - Predict next period using current feature set
    - Record prediction, active regime, and whether re-selection occurred
    """
    n = len(target)
    dates = target.index

    current_regime = int(regime_series.iloc[min_train])
    active_features = regime_selections.get(current_regime, regime_selections['global'])

    results = []
    n_reselections = 0

    for t in range(min_train, n - 1):
        # Current regime (from HMM filtered posterior)
        new_regime = int(regime_series.iloc[t])

        # Check for regime change
        reselected = False
        if new_regime != current_regime:
            # Regime transition detected
            current_regime = new_regime
            active_features = regime_selections.get(current_regime, regime_selections['global'])
            n_reselections += 1
            reselected = True

        # Predict next period using lag-1 features
        if len(active_features) == 0:
            continue

        try:
            X_hist = features.iloc[:t][active_features].dropna()
            y_hist = target.iloc[:t].loc[X_hist.index]

            if len(y_hist) < 20:
                continue

            scaler = StandardScaler()
            X_scaled = scaler.fit_transform(X_hist)

            model = Ridge(alpha=1.0)
            model.fit(X_scaled, y_hist.values)

            X_now = features.iloc[t][active_features].values.reshape(1, -1)
            X_now_scaled = scaler.transform(X_now)
            pred = model.predict(X_now_scaled)[0]
            actual = target.iloc[t + 1]

            results.append({
                'date': dates[t + 1],
                'actual': actual,
                'pred_adaptive': pred,
                'regime': current_regime,
                'reselected': reselected,
                'n_active_features': len(active_features),
            })
        except Exception:
            continue

    print(f'Simulation complete: {len(results)} predictions, {n_reselections} re-selections')
    return pd.DataFrame(results).set_index('date')


# Also run static selection (global features, no regime adaptation)
def simulate_static_selection(
    target: pd.Series,
    features: pd.DataFrame,
    static_features: list,
    min_train: int = 60,
) -> pd.Series:
    """Static feature selection: always use global feature set."""
    n = len(target)
    dates = target.index
    preds = {}

    for t in range(min_train, n - 1):
        try:
            X_hist = features.iloc[:t][static_features].dropna()
            y_hist = target.iloc[:t].loc[X_hist.index]
            if len(y_hist) < 20:
                continue

            scaler = StandardScaler()
            X_s = scaler.fit_transform(X_hist)
            model = Ridge(alpha=1.0)
            model.fit(X_s, y_hist.values)

            X_now = features.iloc[t][static_features].values.reshape(1, -1)
            preds[dates[t + 1]] = model.predict(scaler.transform(X_now))[0]
        except Exception:
            continue

    return pd.Series(preds, name='pred_static')


print('Running adaptive (regime-aware) simulation...')
adaptive_results = simulate_adaptive_selection(
    target_final, features_aligned, regime_aligned,
    posteriors, regime_selections, min_train=60,
    regime_threshold=REGIME_CHANGE_THRESHOLD
)

print('Running static (regime-unaware) simulation...')
static_preds = simulate_static_selection(
    target_final, features_aligned, regime_selections['global'], min_train=60
)

# Merge results
comparison = adaptive_results.join(static_preds, how='inner').dropna()
print(f'Comparison dataset: {len(comparison)} predictions')

---

## 5. Compare Regime-Aware vs Static Feature Selection

In [ ]:
# Purpose: Compare prediction performance: regime-aware vs static
# Key: Regime-aware selection should outperform static in high-vol regimes

actual = comparison['actual']
pred_adaptive = comparison['pred_adaptive']
pred_static = comparison['pred_static']

# Overall MSE
mse_adaptive = mean_squared_error(actual, pred_adaptive)
mse_static = mean_squared_error(actual, pred_static)

# MSE by regime
mse_per_regime = {}
for k in range(N_STATES):
    mask = comparison['regime'] == k
    if mask.sum() < 5:
        continue
    mse_per_regime[k] = {
        'adaptive': mean_squared_error(actual[mask], pred_adaptive[mask]),
        'static': mean_squared_error(actual[mask], pred_static[mask]),
        'n_obs': mask.sum(),
    }

print('Performance Comparison: Regime-Aware vs Static Feature Selection')
print('='*65)
print(f'\nOverall MSE:')
print(f'  Regime-aware (adaptive): {mse_adaptive:.6f}')
print(f'  Static (global features): {mse_static:.6f}')
print(f'  Improvement: {(mse_static - mse_adaptive)/mse_static:.1%}')

print(f'\nMSE by Regime:')
for k, metrics in mse_per_regime.items():
    label = 'Bull' if k == 0 else 'Bear'
    improvement = (metrics['static'] - metrics['adaptive']) / metrics['static']
    print(f'  Regime {k} ({label}, n={metrics["n_obs"]}):')
    print(f'    Adaptive: {metrics["adaptive"]:.6f}')
    print(f'    Static:   {metrics["static"]:.6f}')
    print(f'    Improvement: {improvement:.1%}')

In [ ]:
# Purpose: Comprehensive visualisation of regime-aware vs static results

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

regime_colors_plot = {0: '#2ECC71', 1: '#E74C3C'}

# --- Panel 1: Prediction errors over time ---
err_adaptive = (actual - pred_adaptive).abs()
err_static = (actual - pred_static).abs()

axes[0, 0].plot(comparison.index, err_adaptive.rolling(12).mean(),
                color='steelblue', linewidth=2, label='Adaptive (regime-aware)')
axes[0, 0].plot(comparison.index, err_static.rolling(12).mean(),
                color='darkorange', linewidth=2, linestyle='--', label='Static (global)')

# Shade regime periods
for k in range(N_STATES):
    mask = comparison['regime'] == k
    axes[0, 0].fill_between(comparison.index,
                             0, err_static.rolling(12).mean().max() * 1.1,
                             where=mask, color=regime_colors_plot[k], alpha=0.08)

axes[0, 0].set_ylabel('Rolling 12m MAE')
axes[0, 0].set_title('Prediction Error Over Time (12m Rolling)', fontsize=11)
axes[0, 0].legend(fontsize=9)

# --- Panel 2: Feature score heatmap per regime ---
score_df = pd.DataFrame({
    'Bull (R0)': regime_scores[0],
    'Bear (R1)': regime_scores[1],
    'Global': regime_scores['global'],
})
# Normalise per column
score_df_norm = score_df.div(score_df.max() + 1e-10)

sns.heatmap(
    score_df_norm.T,
    ax=axes[0, 1],
    cmap='YlOrRd',
    linewidths=0.3,
    annot=False,
    cbar_kws={'label': 'Normalised importance'},
)
axes[0, 1].set_title('Feature Importance by Regime\n(normalised within each method)', fontsize=11)
axes[0, 1].set_xlabel('Feature')
axes[0, 1].tick_params(axis='x', rotation=45, labelsize=7)

# --- Panel 3: MSE comparison bar chart by regime ---
regime_labels_plot = []
adaptive_mses = []
static_mses = []
for k, metrics in mse_per_regime.items():
    regime_labels_plot.append(f'Regime {k}\n({"Bull" if k==0 else "Bear"})')
    adaptive_mses.append(metrics['adaptive'])
    static_mses.append(metrics['static'])

x = np.arange(len(regime_labels_plot))
axes[1, 0].bar(x - 0.2, adaptive_mses, 0.4, label='Adaptive', color='#2ECC71', alpha=0.8)
axes[1, 0].bar(x + 0.2, static_mses, 0.4, label='Static', color='#E74C3C', alpha=0.8)
axes[1, 0].set_xticks(x)
axes[1, 0].set_xticklabels(regime_labels_plot)
axes[1, 0].set_ylabel('MSE')
axes[1, 0].set_title('MSE by Regime: Adaptive vs Static', fontsize=11)
axes[1, 0].legend()

# --- Panel 4: Jaccard similarity matrix ---
sets = {'Bull (R0)': set0, 'Bear (R1)': set1, 'Global': set_global}
set_names = list(sets.keys())
jaccard_mat = np.zeros((3, 3))
for i, n1 in enumerate(set_names):
    for j, n2 in enumerate(set_names):
        jaccard_mat[i, j] = jaccard_similarity(sets[n1], sets[n2])

im = axes[1, 1].imshow(jaccard_mat, cmap='Blues', vmin=0, vmax=1)
axes[1, 1].set_xticks(range(3))
axes[1, 1].set_yticks(range(3))
axes[1, 1].set_xticklabels(set_names, rotation=15)
axes[1, 1].set_yticklabels(set_names)
for i in range(3):
    for j in range(3):
        axes[1, 1].text(j, i, f'{jaccard_mat[i,j]:.2f}', ha='center', va='center',
                        fontsize=12, fontweight='bold',
                        color='white' if jaccard_mat[i,j] > 0.6 else 'black')
plt.colorbar(im, ax=axes[1, 1], label='Jaccard similarity')
axes[1, 1].set_title('Jaccard Similarity Between Feature Sets\n(1=identical, 0=no overlap)', fontsize=11)

plt.tight_layout()
plt.savefig('regime_aware_comparison.png', dpi=120, bbox_inches='tight')
plt.show()
print('Figure saved: regime_aware_comparison.png')

---

## Exercise 7.3: PSI Feature Drift Monitor

**Task:** Implement a PSI monitor that computes the Population Stability Index for each feature
comparing its distribution in the first half of the dataset (reference) vs the second half (current).
Identify features with PSI > 0.20 (significant drift).

**Expected output:** DataFrame with PSI for each feature and a flag for drifted features.

**Hint:** Use the `compute_psi` function signature from the guide:
```python
def compute_psi(reference, current, n_bins=10) -> float:
    # Compute quantile bins from reference
    # Count observations per bin in reference and current
    # PSI = sum((cur_pct - ref_pct) * log(cur_pct / ref_pct))
```

In [ ]:
# YOUR CODE HERE
# ---------------

def compute_psi(reference: np.ndarray, current: np.ndarray, n_bins: int = 10) -> float:
    """
    Compute Population Stability Index between reference and current distributions.

    Parameters
    ----------
    reference : np.ndarray — feature values in training/reference period
    current   : np.ndarray — feature values in monitoring period
    n_bins    : int — number of quantile bins

    Returns
    -------
    float : PSI value
    """
    # TODO: Implement PSI calculation
    # Step 1: Compute quantile bin edges from reference
    # Step 2: Count observations in each bin for reference and current
    # Step 3: Convert to percentages (add epsilon to avoid log(0))
    # Step 4: Compute PSI = sum((p_cur - p_ref) * log(p_cur / p_ref))
    pass


# Split data into reference (first half) and current (second half)
n_half = len(features_aligned) // 2
reference_data = features_aligned.iloc[:n_half]
current_data = features_aligned.iloc[n_half:]

print(f'Reference period: {reference_data.index[0].date()} to {reference_data.index[-1].date()}')
print(f'Current period:   {current_data.index[0].date()} to {current_data.index[-1].date()}')

# TODO: Compute PSI for each feature and create a summary DataFrame
# Expected columns: ['feature', 'psi', 'drift_flag']
# where drift_flag = True if psi > 0.20

psi_results = []  # fill this in

print('\nTODO: Compute PSI for each feature using compute_psi() above.')
print('Create psi_results DataFrame with columns: feature, psi, drift_flag')

In [ ]:
# AUTO-GRADED TEST
def test_exercise_73():
    assert compute_psi is not None, 'compute_psi function must be defined'
    # Test with known distributions
    ref = np.random.normal(0, 1, 1000)
    cur_same = np.random.normal(0, 1, 500)  # same distribution
    cur_diff = np.random.normal(2, 1, 500)  # shifted distribution

    psi_same = compute_psi(ref, cur_same)
    psi_diff = compute_psi(ref, cur_diff)

    if psi_same is not None and psi_diff is not None:
        assert psi_same < psi_diff, \
            f'PSI for same distribution ({psi_same:.3f}) should be < shifted ({psi_diff:.3f})'
        assert psi_same < 0.10, f'PSI for same distribution should be small, got {psi_same:.3f}'
        print(f'compute_psi validated: same={psi_same:.3f}, shifted={psi_diff:.3f}')
    else:
        print('compute_psi returns None — complete the implementation.')

    print('Exercise 7.3: Implement compute_psi and run on all features.')

test_exercise_73()

---

## Summary

### Key Takeaways

1. **HMM regime detection** identifies latent market states (bull/bear) from return and volatility data. The Viterbi algorithm provides the most likely state sequence; posterior probabilities enable soft regime assignment.

2. **Regime-specific feature selection** selects different features in each regime. The Jaccard similarity between regimes quantifies how different the optimal sets are — low Jaccard (< 0.4) justifies regime-aware selection.

3. **Adaptive re-selection** is triggered by regime transition detection. The controller monitors HMM posteriors and switches to the appropriate feature set without constant re-fitting.

4. **Regime-aware vs static comparison:** regime-aware selection particularly outperforms static selection in the bear/high-volatility regime, where the predictive features are most different from the full-sample average.

5. **PSI monitoring** detects when a feature's distribution has drifted significantly from its reference distribution — an early warning system for model degradation.

### What's Next

The exercises file (`exercises/01_time_series_exercises.py`) contains three advanced challenges:
- Nonlinear Granger causality via kernel methods
- Building a sklearn-compatible purged CV splitter
- A complete feature drift monitor with PSI, KS, and Wasserstein distance

### References

- Hamilton, J.D. (1989). A New Approach to the Economic Analysis of Nonstationary Time Series. *Econometrica*, 57(2).
- de Prado, M.L. (2018). *Advances in Financial Machine Learning*. Wiley.
- hmmlearn documentation: https://hmmlearn.readthedocs.io/